# 04 — Train: RNN (LSTM/GRU) direct forecaster

Searches a recurrent (LSTM/GRU) architecture for the configured target station's direct multi-horizon water-level forecast over raw, evenly-spaced hourly channel sequences, then evaluates the selected model once on the sealed test cohort.

**Inputs:** joined train/test feature artifacts and their metadata contract
**Outputs:** in-notebook prediction preview/test metrics, an MLflow run hierarchy, and the selected model plus manifest in `models/`

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and loads the joined feature metadata. Unlike Ridge/MLP, this notebook does not search over the engineered-feature ablation subsets: a recurrent model reads raw, evenly-spaced hourly channels instead of a flat engineered vector, so it has no equivalent axis. Instead it searches `cell_type` (GRU vs. LSTM) and `sequence_length` (the lookback window, in hours) alongside `hidden_size` and `num_layers`.

`RnnForecaster` is a stochastic estimator (weight initialization, Adam's optimization path, and the training `DataLoader`'s shuffle order), so `RANDOM_STATE` is fixed and reused in every fold fit and the final retrain as part of the model construction contract. Both the raw channel sequences and the targets are `StandardScaler`-scaled internally by `RnnForecaster`, because gradient-based training needs a zero-centered, unit-scale target to converge.

**Parameters**

The values below are illustrative examples, not run configuration. Imported values from `src/config.py` and executable constants in this notebook are authoritative for a run; Stage-3 feature metadata is authoritative for the realized channel/target contract, and the saved manifest records the fitted model configuration.

| Parameter | Example value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed/joined` | Directory the joined Stage-3 Parquets and metadata are read from. |
| `PREDICTION_PREVIEW_ROWS` | `5` | Number of scored test rows shown in the final preview. |
| `CHANNEL_COLUMNS` | `dataset.feature_subsets["raw_all_stations"]` | The raw per-timestep channels fed to the recurrent encoder at every lookback hour: the target station's `water_level`/`imputed` plus every retained station's raw `water_level`/`imputed`/precipitation/temperature. |
| `TARGET_COLUMNS` | `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_H` | The metadata-declared future water levels predicted directly from one recurrent forward pass. |
| `FORECAST_HORIZON_HOURS` | `24` | Example configured horizon; execution uses the imported value and validates it against the metadata contract width. |
| `CELL_TYPES` | `["gru", "lstm"]` | Candidate recurrent cell types. |
| `SEQUENCE_LENGTHS` | `[24]` | Candidate lookback windows, in hours. |
| `HIDDEN_SIZES` | `[32, 64]` | Candidate recurrent hidden-state widths. |
| `NUM_LAYERS_OPTIONS` | `[1, 2, 3]` | Candidate numbers of stacked recurrent layers. |
| `RNN_DROPOUT` | `0.1` | Fixed inter-layer dropout; a no-op when `num_layers=1`. |
| `RNN_LEARNING_RATE` | `0.001` | Fixed Adam learning rate. |
| `RNN_MAX_EPOCHS` | `50` | Fixed training-epoch budget; there is no early stopping. |
| `RNN_BATCH_SIZE` | `256` | Fixed minibatch size. |
| `RANDOM_STATE` | `src.config.RANDOM_STATE` | Fixed seed for weight initialization and the `DataLoader` shuffle order, reused in every fold fit and the final retrain. |
| `N_VALIDATION_FOLDS` | `5` | Number of expanding-window validation folds. |
| `INITIAL_TRAIN_FRACTION` | `0.50` | Approximate fraction of eligible rows in the first fold's training window. |
| `EMBARGO_HOURS` | `24` | Number of rows left between each fold's training and validation windows. |
| `CV_SELECTION_METRIC` | `"rmse"` | Aggregate CV metric used to select the candidate; `"mae"` is also supported. |
| `MLFLOW_EXPERIMENT_NAME` | `"rnn"` | Experiment receiving the parent, nested fold, and final test runs. |
| `MODEL_PATH` | `models/rnn_{TARGET_STATION_ID}.joblib` | The fitted `RnnForecaster` (torch module, channel scaler, and target scaler) for the selected candidate. |
| `MODEL_METADATA_PATH` | `models/rnn_{TARGET_STATION_ID}.json` | Schema-3.0 model-only manifest recording the estimator, preprocessor, selected architecture, exact channel/target contract, and execution UUID. |

## Joint cell_type/sequence_length/hidden_size/num_layers search

The notebook compares each `(cell_type, sequence_length, hidden_size, num_layers)` quadruple across `N_VALIDATION_FOLDS` expanding-window folds. The candidate count is `len(CELL_TYPES) × len(SEQUENCE_LENGTHS) × len(HIDDEN_SIZES) × len(NUM_LAYERS_OPTIONS)` and the fold-fit count multiplies that by `N_VALIDATION_FOLDS`. For the example values above, that is 12 candidates and 60 fold fits; execution uses the code-cell values.

Unlike Ridge/MLP, where every candidate shares the same eligible cohort and fold indices, here the sequence-eligible cohort — and therefore the folds — differ by `sequence_length` (a longer lookback window discards more early-timeline rows for lack of history). The `sequence_length`-eligible sequences and folds are built once per `sequence_length` and cached, then reused across the `len(CELL_TYPES) × len(HIDDEN_SIZES) × len(NUM_LAYERS_OPTIONS)` candidates that share it.

In [ ]:
import json
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import dump
from tqdm.auto import tqdm

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    RANDOM_STATE,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset, time_series_splits
from src.metrics import metric_tables
from src.plots import (
    cv_error_boxplots_figure,
    predicted_vs_actual_figure,
    test_error_boxplots_figure,
)
from src.regime_persistence import (
    regime_mlflow_metrics,
    regime_mlflow_params,
    sealed_test_regime_tables,
)
from src.training import prediction_preview, summarize_cv_metrics, validate_predictions

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
NOTEBOOK_EXECUTION_UUID = str(uuid4())
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
CELL_TYPES: list[str] = ["gru", "lstm"]
SEQUENCE_LENGTHS = [24]
HIDDEN_SIZES = [32, 64]
NUM_LAYERS_OPTIONS = [1, 2, 3]
RNN_DROPOUT = 0.1
RNN_LEARNING_RATE = 0.001
RNN_MAX_EPOCHS = 50
RNN_BATCH_SIZE = 256
MLFLOW_EXPERIMENT_NAME = "rnn"
PREDICTION_PREVIEW_ROWS = 5
if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")
station_id = TARGET_STATION_ID
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / f"rnn_{station_id}.joblib"
MODEL_METADATA_PATH = MODEL_DIR / f"rnn_{station_id}.json"

## Shared evaluation cohort

The model is fit and scored on rows from the joined feature artifacts, narrowed twice:

1. **Stage 3 marked the future window valid, with complete predictors.** Same rule as Ridge/MLP/XGBoost/Random Forest/Extra Trees: `{TARGET_STATION_ID}__target_valid` is true, and every full-contract predictor and target is present at issue time `t`.
2. **A complete `sequence_length`-hour raw channel window exists ending at `t`, within the same split.** A row keeps rule 1's eligibility only if `sequence_length` consecutive hours of finite raw channel values (`CHANNEL_COLUMNS`) exist immediately before and including it, sourced from a raw contiguous per-timestep frame independent of rule 1's row filtering. Imputed-but-present hours inside the window are kept — the per-station `imputed` flag is itself one of the channels, so the model can learn to weight accordingly. This narrowing is `sequence_length`-specific and never reaches across the train/test split boundary.

## Shared helpers

The joined dataset and chronological folds come from `src.dataset`; prediction checks, metric summaries, and previews come from `src.training`; the evaluation figures from `src.plots`. Raw-channel loading, sequence construction, the `RnnForecaster` wrapper, and RNN candidate ranking are RNN-specific and come from `src.rnn`.

In [ ]:
from src.rnn import (
    build_rnn_estimator,
    build_sequences,
    load_raw_channel_frame,
    save_rnn_manifest,
    select_candidate,
)

## Load the joined dataset

`load_joined_dataset()` reads the joined feature metadata and both Parquet artifacts, applies the shared eligibility cohort (rule 1 above) to each split independently, and derives the ordered feature subsets — including `raw_all_stations`, this notebook's channel contract.

In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=station_id,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
contract = dataset.contract
TARGET_COLUMNS = list(contract.target_columns)
CHANNEL_COLUMNS = dataset.feature_subsets["raw_all_stations"]
train_rows = dataset.train_rows
test_rows = dataset.test_rows
INPUT_PARQUET_SHA256_PARAMS = dataset.input_hashes

## Load raw channel frames

`load_raw_channel_frame()` reads only `timestamp` plus `CHANNEL_COLUMNS` from each joined Parquet, independent of `load_joined_dataset`'s row-level eligibility filtering, and validates the result is a unique, ascending, contiguous hourly grid. This is what lets a sequence lookback window reach hours that fail rule 1's target-valid/complete-predictor gate.

In [ ]:
train_raw_channel_frame = load_raw_channel_frame(
    train_path, channel_columns=CHANNEL_COLUMNS
)
test_raw_channel_frame = load_raw_channel_frame(
    test_path, channel_columns=CHANNEL_COLUMNS
)

## Joint cell_type/sequence_length/hidden_size/num_layers search

Each `(cell_type, sequence_length, hidden_size, num_layers)` candidate has one MLflow parent and each fold has one nested child run. The execution must produce the complete `len(CELL_TYPES) × len(SEQUENCE_LENGTHS) × len(HIDDEN_SIZES) × len(NUM_LAYERS_OPTIONS)` Cartesian product before selection, for that candidate count multiplied by `N_VALIDATION_FOLDS` fold fits.

Every fold builds a fresh `RnnForecaster` via `build_rnn_estimator` — a channel- and target-`StandardScaler`-scaled GRU/LSTM trained from scratch — using only that fold's training sequences and the fixed `RANDOM_STATE`, so every fold's weight initialization, shuffle order, and optimization path is reproducible. The sealed test cohort is not referenced until the final fit below.

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cv_results_rows = []
cv_horizon_rows_by_candidate = {}
sequences_by_length = {}
expected_candidate_keys = {
    (cell_type, sequence_length, hidden_size, num_layers)
    for cell_type in CELL_TYPES
    for sequence_length in SEQUENCE_LENGTHS
    for hidden_size in HIDDEN_SIZES
    for num_layers in NUM_LAYERS_OPTIONS
}
total_fold_fits = len(expected_candidate_keys) * N_VALIDATION_FOLDS
search_progress = tqdm(total=total_fold_fits, desc="RNN CV search", unit="fit")


for sequence_length in SEQUENCE_LENGTHS:
    seq_train_sequences, seq_train_targets, seq_train_origins = build_sequences(
        train_raw_channel_frame,
        train_rows,
        sequence_length=sequence_length,
        channel_columns=CHANNEL_COLUMNS,
        target_columns=TARGET_COLUMNS,
        artifact_name="train",
    )
    seq_folds, seq_validation_test_size = time_series_splits(
        len(seq_train_origins),
        initial_train_fraction=INITIAL_TRAIN_FRACTION,
        n_validation_folds=N_VALIDATION_FOLDS,
        embargo_rows=EMBARGO_HOURS,
    )
    sequences_by_length[sequence_length] = {
        "train_sequences": seq_train_sequences,
        "train_targets": seq_train_targets,
        "train_origins": seq_train_origins,
        "folds": seq_folds,
        "validation_test_size": seq_validation_test_size,
    }

    train_sequences = seq_train_sequences
    train_targets = seq_train_targets
    train_origins = seq_train_origins
    cv_splits = seq_folds
    validation_test_size = seq_validation_test_size

    for cell_type in CELL_TYPES:
        for hidden_size in HIDDEN_SIZES:
            for num_layers in NUM_LAYERS_OPTIONS:
                fold_aggregate_rows = []
                fold_horizon_rows = []
                with mlflow.start_run(
                    run_name=(
                        f"rnn_cv_{cell_type}_{sequence_length}_{hidden_size}_{num_layers}"
                    ),
                    nested=False,
                    tags={
                        "phase": "cv",
                        "run_type": "candidate_parent",
                        "cell_type": cell_type,
                        "sequence_length": str(sequence_length),
                        "hidden_size": str(hidden_size),
                        "num_layers": str(num_layers),
                        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                    },
                ):
                    mlflow.log_params(
                        {
                            "phase": "cv",
                            "run_type": "candidate_parent",
                            "cell_type": cell_type,
                            "sequence_length": sequence_length,
                            "hidden_size": hidden_size,
                            "num_layers": num_layers,
                            "dropout": RNN_DROPOUT,
                            "learning_rate": RNN_LEARNING_RATE,
                            "max_epochs": RNN_MAX_EPOCHS,
                            "batch_size": RNN_BATCH_SIZE,
                            "random_state": RANDOM_STATE,
                            **INPUT_PARQUET_SHA256_PARAMS,
                            "channel_count": len(CHANNEL_COLUMNS),
                            "channel_columns": json.dumps(CHANNEL_COLUMNS),
                            "n_validation_folds": N_VALIDATION_FOLDS,
                            "validation_test_size": validation_test_size,
                            "embargo_hours": EMBARGO_HOURS,
                            "selection_metric": CV_SELECTION_METRIC,
                            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                            "sequence_eligible_train_rows": len(train_origins),
                        }
                    )

                    for fold_number, (
                        fold_train_indices,
                        fold_validation_indices,
                    ) in enumerate(cv_splits, start=1):
                        with mlflow.start_run(
                            run_name=(
                                f"rnn_cv_{cell_type}_{sequence_length}_{hidden_size}_"
                                f"{num_layers}_fold_{fold_number}"
                            ),
                            nested=True,
                            tags={
                                "phase": "cv",
                                "run_type": "fold",
                                "cell_type": cell_type,
                                "sequence_length": str(sequence_length),
                                "hidden_size": str(hidden_size),
                                "num_layers": str(num_layers),
                                "fold": str(fold_number),
                                "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                            },
                        ):
                            search_progress.set_description(
                                f"RNN CV search ({cell_type}, sequence_length={sequence_length}, "
                                f"hidden_size={hidden_size}, num_layers={num_layers}, "
                                f"fold {fold_number})"
                            )
                            fold_model = build_rnn_estimator(
                                cell_type=cell_type,
                                sequence_length=sequence_length,
                                hidden_size=hidden_size,
                                num_layers=num_layers,
                                dropout=RNN_DROPOUT,
                                learning_rate=RNN_LEARNING_RATE,
                                max_epochs=RNN_MAX_EPOCHS,
                                batch_size=RNN_BATCH_SIZE,
                                random_state=RANDOM_STATE,
                            )
                            fold_model.fit(
                                train_sequences[fold_train_indices],
                                train_targets[fold_train_indices],
                            )
                            fold_predictions = validate_predictions(
                                fold_model.predict(
                                    train_sequences[fold_validation_indices]
                                ),
                                expected_rows=len(fold_validation_indices),
                                target_columns=TARGET_COLUMNS,
                                artifact_name="fold",
                            )
                            fold_aggregate, fold_per_horizon = metric_tables(
                                pd.DataFrame(
                                    train_targets[fold_validation_indices],
                                    columns=TARGET_COLUMNS,
                                ),
                                fold_predictions,
                                target_columns=TARGET_COLUMNS,
                                station_id=station_id,
                            )
                            fold_aggregate_rows.append(fold_aggregate.iloc[0])
                            fold_horizon_rows.append(fold_per_horizon)
                            search_progress.update(1)

                            fold_train_timestamps = train_origins["timestamp"].iloc[
                                fold_train_indices
                            ]
                            fold_validation_timestamps = train_origins[
                                "timestamp"
                            ].iloc[fold_validation_indices]
                            mlflow.log_params(
                                {
                                    "phase": "cv",
                                    "run_type": "fold",
                                    "cell_type": cell_type,
                                    "sequence_length": sequence_length,
                                    "hidden_size": hidden_size,
                                    "num_layers": num_layers,
                                    "fold": fold_number,
                                    **INPUT_PARQUET_SHA256_PARAMS,
                                    "train_rows": len(fold_train_indices),
                                    "validation_rows": len(fold_validation_indices),
                                    "gap_rows": EMBARGO_HOURS,
                                    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                                    "train_start": fold_train_timestamps.iloc[
                                        0
                                    ].isoformat(),
                                    "train_end": fold_train_timestamps.iloc[
                                        -1
                                    ].isoformat(),
                                    "validation_start": fold_validation_timestamps.iloc[
                                        0
                                    ].isoformat(),
                                    "validation_end": fold_validation_timestamps.iloc[
                                        -1
                                    ].isoformat(),
                                    "train_index_start": int(fold_train_indices[0]),
                                    "train_index_end": int(fold_train_indices[-1]),
                                    "validation_index_start": int(
                                        fold_validation_indices[0]
                                    ),
                                    "validation_index_end": int(
                                        fold_validation_indices[-1]
                                    ),
                                }
                            )
                            mlflow.log_metrics(
                                {
                                    "fold_mae": float(fold_aggregate.iloc[0]["mae"]),
                                    "fold_rmse": float(fold_aggregate.iloc[0]["rmse"]),
                                    "fold_me": float(fold_aggregate.iloc[0]["me"]),
                                    "fold_r2": float(fold_aggregate.iloc[0]["r2"]),
                                    **{
                                        f"fold_mae_horizon_{row.horizon_hours:02d}": float(
                                            row.mae
                                        )
                                        for row in fold_per_horizon.itertuples()
                                    },
                                    **{
                                        f"fold_me_horizon_{row.horizon_hours:02d}": float(
                                            row.me
                                        )
                                        for row in fold_per_horizon.itertuples()
                                    },
                                    **{
                                        f"fold_r2_horizon_{row.horizon_hours:02d}": float(
                                            row.r2
                                        )
                                        for row in fold_per_horizon.itertuples()
                                    },
                                    **{
                                        f"fold_rmse_horizon_{row.horizon_hours:02d}": float(
                                            row.rmse
                                        )
                                        for row in fold_per_horizon.itertuples()
                                    },
                                }
                            )

                    fold_aggregate_metrics = pd.DataFrame(fold_aggregate_rows)
                    fold_horizon_metrics = pd.concat(
                        fold_horizon_rows, ignore_index=True
                    )
                    parent_metrics = summarize_cv_metrics(
                        fold_aggregate_metrics, fold_horizon_metrics
                    )
                    candidate_key = (
                        cell_type,
                        sequence_length,
                        hidden_size,
                        num_layers,
                    )
                    cv_horizon_rows_by_candidate[candidate_key] = (
                        fold_horizon_rows.copy()
                    )
                    mlflow.log_metrics(parent_metrics)
                    cv_results_rows.append(
                        {
                            "cell_type": cell_type,
                            "sequence_length": sequence_length,
                            "hidden_size": hidden_size,
                            "num_layers": num_layers,
                            "dropout": RNN_DROPOUT,
                            "sequence_eligible_train_rows": len(train_origins),
                            "mae_mean": parent_metrics["cv_mae_mean"],
                            "mae_std": parent_metrics["cv_mae_std"],
                            "rmse_mean": parent_metrics["cv_rmse_mean"],
                            "rmse_std": parent_metrics["cv_rmse_std"],
                            "me_mean": parent_metrics["cv_me_mean"],
                            "me_std": parent_metrics["cv_me_std"],
                            "r2_mean": parent_metrics["cv_r2_mean"],
                            "r2_std": parent_metrics["cv_r2_std"],
                            **{
                                metric_name: metric_value
                                for metric_name, metric_value in parent_metrics.items()
                                if metric_name
                                not in {
                                    "cv_mae_mean",
                                    "cv_mae_std",
                                    "cv_rmse_mean",
                                    "cv_rmse_std",
                                    "cv_me_mean",
                                    "cv_me_std",
                                    "cv_r2_mean",
                                    "cv_r2_std",
                                }
                            },
                        }
                    )

search_progress.close()
cv_experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if cv_experiment is None:
    raise ValueError(f"MLflow experiment {MLFLOW_EXPERIMENT_NAME!r} was not found")
current_cv_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'candidate_parent'"
    ),
)
current_fold_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'fold'"
    ),
)
parent_keys = {
    (
        str(row["tags.cell_type"]),
        int(row["tags.sequence_length"]),
        int(row["tags.hidden_size"]),
        int(row["tags.num_layers"]),
    )
    for _, row in current_cv_runs.iterrows()
}
if (
    len(current_cv_runs) != len(expected_candidate_keys)
    or parent_keys != expected_candidate_keys
):
    raise ValueError(
        "Current execution must produce the complete 12-candidate cell_type/sequence_length/"
        "hidden_size/num_layers product: expected "
        f"{len(expected_candidate_keys)} {sorted(expected_candidate_keys)}, "
        f"got {len(current_cv_runs)} {sorted(parent_keys)}"
    )
expected_fold_count = len(expected_candidate_keys) * N_VALIDATION_FOLDS
if len(current_fold_runs) != expected_fold_count:
    raise ValueError(
        f"Current execution must produce {expected_fold_count} nested fold runs, "
        f"got {len(current_fold_runs)}"
    )
fold_keys = {
    (
        str(row["tags.cell_type"]),
        int(row["tags.sequence_length"]),
        int(row["tags.hidden_size"]),
        int(row["tags.num_layers"]),
        int(row["tags.fold"]),
    )
    for _, row in current_fold_runs.iterrows()
}
expected_fold_keys = {
    (cell_type, sequence_length, hidden_size, num_layers, fold_number)
    for cell_type, sequence_length, hidden_size, num_layers in expected_candidate_keys
    for fold_number in range(1, N_VALIDATION_FOLDS + 1)
}
if fold_keys != expected_fold_keys:
    raise ValueError(
        "Current execution fold runs do not cover every candidate and fold"
    )
cv_results = pd.DataFrame(cv_results_rows)
if len(cv_results) != len(expected_candidate_keys):
    raise ValueError("The in-memory CV result table is incomplete")
if (
    set(
        zip(
            cv_results["cell_type"],
            cv_results["sequence_length"],
            cv_results["hidden_size"],
            cv_results["num_layers"],
        )
    )
    != expected_candidate_keys
):
    raise ValueError(
        "The in-memory CV result table does not match the candidate product"
    )
cv_results = cv_results.sort_values(
    ["cell_type", "sequence_length", "hidden_size", "num_layers"], kind="stable"
).reset_index(drop=True)
(
    selected_cell_type,
    selected_sequence_length,
    selected_hidden_size,
    selected_num_layers,
) = select_candidate(cv_results, CV_SELECTION_METRIC)
fold_horizon_rows = cv_horizon_rows_by_candidate[
    (
        selected_cell_type,
        selected_sequence_length,
        selected_hidden_size,
        selected_num_layers,
    )
]
print(
    f"Selected RNN candidate by CV {CV_SELECTION_METRIC.upper()}: "
    f"cell_type={selected_cell_type!r}, sequence_length={selected_sequence_length}, "
    f"hidden_size={selected_hidden_size}, num_layers={selected_num_layers}"
)
display(
    cv_results[
        [
            "cell_type",
            "sequence_length",
            "hidden_size",
            "num_layers",
            "dropout",
            "sequence_eligible_train_rows",
            "mae_mean",
            "mae_std",
            "rmse_mean",
            "rmse_std",
            "me_mean",
            "me_std",
            "r2_mean",
            "r2_std",
        ]
    ]
)

## Retrain the selected candidate

The selected `(cell_type, sequence_length, hidden_size, num_layers)` quadruple is retrained once on all sequence-eligible, chronologically ordered training rows for its `sequence_length`, using the same `build_rnn_estimator` construction path — and the same fixed `RANDOM_STATE` — as every CV fold. The model is held in memory here; it is written to disk together with its manifest only after the sealed-test cell below succeeds, so a crashed run leaves the previous artifacts intact.

In [ ]:
selected_cache = sequences_by_length[selected_sequence_length]
train_sequences = selected_cache["train_sequences"]
train_targets = selected_cache["train_targets"]
train_origins = selected_cache["train_origins"]

final_model = build_rnn_estimator(
    cell_type=selected_cell_type,
    sequence_length=selected_sequence_length,
    hidden_size=selected_hidden_size,
    num_layers=selected_num_layers,
    dropout=RNN_DROPOUT,
    learning_rate=RNN_LEARNING_RATE,
    max_epochs=RNN_MAX_EPOCHS,
    batch_size=RNN_BATCH_SIZE,
    random_state=RANDOM_STATE,
)
final_model.fit(train_sequences, train_targets)
print(
    f"Fitted the selected RNN candidate on {len(train_origins):,} sequence-eligible "
    f"training rows (sequence_length={selected_sequence_length})."
)

## Evaluate on the sealed test cohort

Rebuilding sequences for the sealed test cohort narrows it further: only rows with a complete `selected_sequence_length`-hour raw channel window are scored, so the excluded warm-up count is reported explicitly for transparency. Every metric, figure, and manifest call below uses `test_origins` — the sequence-eligible subset — not `test_rows`.

The model and its model-only manifest are written at the end of this cell; CV and sealed-test diagnostics are logged to MLflow and remain available in memory for the analysis below.

In [ ]:
test_sequences, test_targets, test_origins = build_sequences(
    test_raw_channel_frame,
    test_rows,
    sequence_length=selected_sequence_length,
    channel_columns=CHANNEL_COLUMNS,
    target_columns=TARGET_COLUMNS,
    artifact_name="test",
)
excluded_test_rows = len(test_rows) - len(test_origins)
print(
    f"Excluded {excluded_test_rows:,} of {len(test_rows):,} sealed-test rows without a "
    f"complete {selected_sequence_length}-hour raw channel window."
)
test_predictions = validate_predictions(
    final_model.predict(test_sequences),
    expected_rows=len(test_origins),
    target_columns=TARGET_COLUMNS,
    artifact_name="test",
)

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_origins[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=station_id,
)
regime_definition, regime_aggregate_metrics, regime_horizon_metrics = (
    sealed_test_regime_tables(
        test_origins[TARGET_COLUMNS],
        test_predictions,
        target_columns=TARGET_COLUMNS,
        station_id=station_id,
        quartile_cutoffs_cm=dataset.target_water_level_quartile_cutoffs_cm,
        quartile_reference_count=dataset.target_water_level_quartile_reference_count,
    )
)
if not np.isfinite(aggregate_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("RNN reported non-finite aggregate metrics")
if not np.isfinite(per_horizon_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("RNN reported non-finite horizon metrics")
with mlflow.start_run(
    run_name=(
        f"rnn_test_{selected_cell_type}_{selected_sequence_length}_"
        f"{selected_hidden_size}_{selected_num_layers}"
    ),
    nested=False,
    tags={
        "phase": "test",
        "run_type": "sealed_test",
        "cell_type": selected_cell_type,
        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    },
):
    mlflow.log_params(
        {
            "phase": "test",
            "run_type": "sealed_test",
            "cell_type": selected_cell_type,
            "sequence_length": selected_sequence_length,
            "hidden_size": selected_hidden_size,
            "num_layers": selected_num_layers,
            "dropout": RNN_DROPOUT,
            **INPUT_PARQUET_SHA256_PARAMS,
            "channel_count": len(CHANNEL_COLUMNS),
            "channel_columns": json.dumps(CHANNEL_COLUMNS),
            "selection_metric": CV_SELECTION_METRIC,
            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
            "cv_selected_metric": float(
                cv_results.loc[
                    cv_results["cell_type"].eq(selected_cell_type)
                    & cv_results["sequence_length"].eq(selected_sequence_length)
                    & cv_results["hidden_size"].eq(selected_hidden_size)
                    & cv_results["num_layers"].eq(selected_num_layers),
                    f"{CV_SELECTION_METRIC}_mean",
                ].iloc[0]
            ),
            **regime_mlflow_params(regime_definition),
            "scored_issue_times": len(test_origins),
            "excluded_test_rows": excluded_test_rows,
        }
    )
    mlflow.log_metrics(
        {
            **regime_mlflow_metrics(regime_aggregate_metrics, regime_horizon_metrics),
            "test_mae": float(aggregate_metrics.iloc[0]["mae"]),
            "test_rmse": float(aggregate_metrics.iloc[0]["rmse"]),
            "test_me": float(aggregate_metrics.iloc[0]["me"]),
            "test_r2": float(aggregate_metrics.iloc[0]["r2"]),
            **{
                f"test_mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_me_horizon_{row.horizon_hours:02d}": float(row.me)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_r2_horizon_{row.horizon_hours:02d}": float(row.r2)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                for row in per_horizon_metrics.itertuples()
            },
        }
    )
    cv_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
    cv_rmse_mae_boxplots_fig = cv_error_boxplots_figure(
        cv_horizon_metrics,
        TARGET_COLUMNS,
        title=(
            f"RNN CV errors — {selected_cell_type}, "
            f"sequence_length={selected_sequence_length}, "
            f"hidden_size={selected_hidden_size}, num_layers={selected_num_layers}"
        ),
    )
    mlflow.log_figure(cv_rmse_mae_boxplots_fig, "cv_rmse_mae_boxplots.png")
    plt.show()
    plt.close(cv_rmse_mae_boxplots_fig)
    test_error_boxplots_fig = test_error_boxplots_figure(
        test_origins,
        test_predictions,
        per_horizon_metrics,
        TARGET_COLUMNS,
        title=(
            f"RNN final-test errors — {selected_cell_type}, "
            f"sequence_length={selected_sequence_length}, "
            f"hidden_size={selected_hidden_size}, num_layers={selected_num_layers}"
        ),
    )
    mlflow.log_figure(test_error_boxplots_fig, "test_error_boxplots.png")
    plt.show()
    plt.close(test_error_boxplots_fig)
    test_predicted_vs_actual_fig = predicted_vs_actual_figure(
        test_origins[TARGET_COLUMNS],
        test_predictions,
        TARGET_COLUMNS,
        title=(
            f"RNN predicted vs actual — {selected_cell_type}, "
            f"sequence_length={selected_sequence_length}, "
            f"hidden_size={selected_hidden_size}, num_layers={selected_num_layers}"
        ),
    )
    mlflow.log_figure(test_predicted_vs_actual_fig, "test_predicted_vs_actual.png")
    plt.show()
    plt.close(test_predicted_vs_actual_fig)
    final_retrain_loss_curve_fig, final_retrain_loss_curve_axis = plt.subplots(
        figsize=(8, 4)
    )
    final_retrain_loss_curve_axis.plot(
        range(1, len(final_model.epoch_losses_) + 1), final_model.epoch_losses_
    )
    final_retrain_loss_curve_axis.set_xlabel("Epoch")
    final_retrain_loss_curve_axis.set_ylabel("Training MSE (scaled targets)")
    final_retrain_loss_curve_axis.set_title(
        f"RNN final retrain loss curve — {selected_cell_type}, "
        f"sequence_length={selected_sequence_length}, "
        f"hidden_size={selected_hidden_size}, num_layers={selected_num_layers}"
    )
    final_retrain_loss_curve_axis.grid(alpha=0.3)
    mlflow.log_figure(final_retrain_loss_curve_fig, "final_retrain_loss_curve.png")
    plt.show()
    plt.close(final_retrain_loss_curve_fig)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
dump(final_model, MODEL_PATH)
save_rnn_manifest(
    MODEL_METADATA_PATH,
    model_path=MODEL_PATH,
    execution_uuid=NOTEBOOK_EXECUTION_UUID,
    contract=contract,
    channel_columns=CHANNEL_COLUMNS,
    selected_cell_type=selected_cell_type,
    selected_sequence_length=selected_sequence_length,
    selected_hidden_size=selected_hidden_size,
    selected_num_layers=selected_num_layers,
    fixed_dropout=RNN_DROPOUT,
)
print(f"Saved RNN model to {MODEL_PATH}")
print(f"Saved RNN model manifest to {MODEL_METADATA_PATH}")
print(
    f"RNN test results for {station_id} (selected cell_type={selected_cell_type!r}, "
    f"sequence_length={selected_sequence_length}, hidden_size={selected_hidden_size}, "
    f"num_layers={selected_num_layers})"
)
display(aggregate_metrics)
display(per_horizon_metrics)
display(
    prediction_preview(
        test_origins,
        test_predictions,
        target_columns=TARGET_COLUMNS,
    ).head(PREDICTION_PREVIEW_ROWS)
)

# RNN saved-model evaluation

This section validates the saved RNN model manifest and presents the current execution's in-memory CV and sealed-test diagnostics. Results are stored in MLflow, not in the model manifest.

## Load the saved RNN execution record

The joined dataset, raw channel frames, and model manifest are loaded here to validate the saved model against the current station, horizon, target columns, and channel columns. The comparison tables also use the current execution's in-memory `cv_results`, `aggregate_metrics`, and `per_horizon_metrics`; results are deliberately not read from the manifest.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.plots import forecast_window_figures
from src.rnn import load_raw_channel_frame, load_rnn_manifest, score_saved_model

if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")

COMPARISON_PROCESSED_DIR = Path("data/processed/joined")
COMPARISON_METADATA_PATH = (
    COMPARISON_PROCESSED_DIR / "all_stations_feature_metadata.json"
)
COMPARISON_TRAIN_PATH = COMPARISON_PROCESSED_DIR / "all_stations_train_features.parquet"
COMPARISON_TEST_PATH = COMPARISON_PROCESSED_DIR / "all_stations_test_features.parquet"
COMPARISON_MODEL_PATH = Path("models") / f"rnn_{TARGET_STATION_ID}.joblib"
COMPARISON_MODEL_METADATA_PATH = Path("models") / f"rnn_{TARGET_STATION_ID}.json"

comparison_dataset = load_joined_dataset(
    COMPARISON_METADATA_PATH,
    COMPARISON_TRAIN_PATH,
    COMPARISON_TEST_PATH,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
comparison_contract = comparison_dataset.contract
COMPARISON_TARGET_COLUMNS = list(comparison_contract.target_columns)
COMPARISON_CHANNEL_COLUMNS = comparison_dataset.feature_subsets["raw_all_stations"]
comparison_test_rows = comparison_dataset.test_rows
comparison_raw_test_channel_frame = load_raw_channel_frame(
    COMPARISON_TEST_PATH, channel_columns=COMPARISON_CHANNEL_COLUMNS
)

rnn_manifest = load_rnn_manifest(
    COMPARISON_MODEL_METADATA_PATH,
    contract=comparison_contract,
    channel_columns=COMPARISON_CHANNEL_COLUMNS,
)

## Inspect the recorded execution

The manifest is written once, after the sealed test has been scored, so an execution that crashed part-way leaves no record and the previous manifest survives untouched. The `execution_uuid` below is the same tag the MLflow runs of that execution carry, which is how the two views are tied together.

In [ ]:
selected_candidate_table = cv_results
selected_horizon_metrics = per_horizon_metrics.rename(
    columns={metric: f"test_{metric}" for metric in ("mae", "rmse", "me", "r2")}
)
selected_candidate = selected_candidate_table.loc[
    selected_candidate_table["cell_type"].eq(rnn_manifest.selected_cell_type)
    & selected_candidate_table["sequence_length"].eq(
        rnn_manifest.selected_sequence_length
    )
    & selected_candidate_table["hidden_size"].eq(rnn_manifest.selected_hidden_size)
    & selected_candidate_table["num_layers"].eq(rnn_manifest.selected_num_layers)
].iloc[0]
selected_execution_summary = pd.DataFrame(
    [
        {
            "execution_uuid": rnn_manifest.execution_uuid,
            "manifest": str(COMPARISON_MODEL_METADATA_PATH),
            "candidate_count": len(selected_candidate_table),
            "selection_metric": CV_SELECTION_METRIC,
            "selected_cell_type": rnn_manifest.selected_cell_type,
            "selected_sequence_length": rnn_manifest.selected_sequence_length,
            "selected_hidden_size": rnn_manifest.selected_hidden_size,
            "selected_num_layers": rnn_manifest.selected_num_layers,
        }
    ]
)
display(selected_execution_summary)

## Compare cross-validation candidates

Candidate ranking uses only the recorded CV metrics and the configured selection metric; the sealed-test metrics are not used to rank candidates.

In [ ]:
candidate_columns = [
    "cell_type",
    "sequence_length",
    "hidden_size",
    "num_layers",
    "dropout",
    "sequence_eligible_train_rows",
    "mae_mean",
    "mae_std",
    "rmse_mean",
    "rmse_std",
    "me_mean",
    "me_std",
    "r2_mean",
    "r2_std",
]
candidate_comparison_table = selected_candidate_table[candidate_columns].copy()
top5_by_rmse_table = (
    candidate_comparison_table.sort_values("rmse_mean").head(5).reset_index(drop=True)
)
display(top5_by_rmse_table)
display(candidate_comparison_table)

## Visualize cross-validation error

The heatmap shows CV RMSE across `hidden_size` and `num_layers`, faceted by `cell_type` and `sequence_length` (every recorded candidate gets exactly one cell). The line chart adds fold-to-fold RMSE variation as error bars, with `sequence_length` on the x-axis.

In [ ]:
cv_rmse_heatmap_zmin = candidate_comparison_table["rmse_mean"].min()
cv_rmse_heatmap_zmax = candidate_comparison_table["rmse_mean"].max()
cv_rmse_heatmap_figure = make_subplots(
    rows=len(CELL_TYPES),
    cols=len(SEQUENCE_LENGTHS),
    subplot_titles=[
        f"{cell_type} · sequence_length={sequence_length}"
        for cell_type in CELL_TYPES
        for sequence_length in SEQUENCE_LENGTHS
    ],
    shared_yaxes=True,
)
for row_index, cell_type in enumerate(CELL_TYPES, start=1):
    for col_index, sequence_length in enumerate(SEQUENCE_LENGTHS, start=1):
        facet_table = candidate_comparison_table[
            candidate_comparison_table["cell_type"].eq(cell_type)
            & candidate_comparison_table["sequence_length"].eq(sequence_length)
        ]
        facet_heatmap_values = (
            facet_table.pivot(
                index="num_layers", columns="hidden_size", values="rmse_mean"
            )
            .sort_index(axis=0)
            .sort_index(axis=1)
        )
        cv_rmse_heatmap_figure.add_trace(
            go.Heatmap(
                z=facet_heatmap_values.to_numpy(),
                x=list(map(str, facet_heatmap_values.columns.tolist())),
                y=list(map(str, facet_heatmap_values.index.tolist())),
                zmin=cv_rmse_heatmap_zmin,
                zmax=cv_rmse_heatmap_zmax,
                coloraxis="coloraxis",
                hovertemplate=(
                    "hidden_size=%{x}<br>num_layers=%{y}<br>CV RMSE=%{z:.4f}<extra></extra>"
                ),
            ),
            row=row_index,
            col=col_index,
        )
cv_rmse_heatmap_figure.update_layout(
    title="RNN candidate CV RMSE by cell_type, sequence_length, hidden_size, and num_layers",
    coloraxis={"colorscale": "Viridis", "colorbar": {"title": "CV RMSE"}},
)
cv_rmse_heatmap_figure.update_xaxes(title_text="hidden_size")
cv_rmse_heatmap_figure.update_yaxes(title_text="num_layers", col=1)
display(cv_rmse_heatmap_figure)

In [ ]:
cv_rmse_by_sequence_length_figure = go.Figure()
for (
    cell_type,
    hidden_size,
    num_layers,
), candidate_group in candidate_comparison_table.groupby(
    ["cell_type", "hidden_size", "num_layers"], sort=True
):
    candidate_group = candidate_group.sort_values("sequence_length")
    cv_rmse_by_sequence_length_figure.add_trace(
        go.Scatter(
            x=candidate_group["sequence_length"],
            y=candidate_group["rmse_mean"],
            mode="lines+markers",
            name=f"{cell_type} · hidden_size={hidden_size} · num_layers={num_layers}",
            error_y={
                "type": "data",
                "array": candidate_group["rmse_std"],
                "visible": True,
            },
        )
    )
cv_rmse_by_sequence_length_figure.update_layout(
    title="RNN CV RMSE versus sequence_length, by cell_type, hidden_size, and num_layers",
    xaxis_title="Sequence length (hours)",
    yaxis_title="CV RMSE",
)
display(cv_rmse_by_sequence_length_figure)

## Inspect selected-candidate sealed-test performance

These values belong only to the candidate recorded by the selected sealed-test run. The final chart shows its MAE and RMSE at each available forecast horizon.

In [ ]:
selected_candidate_sealed_test_summary = pd.DataFrame(
    [
        {
            "execution_uuid": rnn_manifest.execution_uuid,
            "cell_type": selected_candidate["cell_type"],
            "sequence_length": selected_candidate["sequence_length"],
            "hidden_size": selected_candidate["hidden_size"],
            "num_layers": selected_candidate["num_layers"],
            "cv_mae_mean": selected_candidate["mae_mean"],
            "cv_rmse_mean": selected_candidate["rmse_mean"],
            "cv_me_mean": selected_candidate["me_mean"],
            "cv_r2_mean": selected_candidate["r2_mean"],
            **{
                f"test_{metric}": float(aggregate_metrics.iloc[0][metric])
                for metric in ("mae", "rmse", "me", "r2")
            },
        }
    ]
)
display(selected_candidate_sealed_test_summary)

sealed_test_horizon_figure = go.Figure(
    [
        go.Scatter(
            x=selected_horizon_metrics["horizon_hours"],
            y=selected_horizon_metrics[metric_name],
            mode="lines+markers",
            name=metric_name.upper(),
        )
        for metric_name in ("test_mae", "test_rmse", "test_me", "test_r2")
    ]
)
sealed_test_horizon_figure.update_layout(
    title="Selected RNN candidate sealed-test error by horizon",
    xaxis_title="Forecast horizon (hours)",
    yaxis_title="Error",
)
display(sealed_test_horizon_figure)

## Reload and score the saved RNN model

Reloads the saved RNN model, rebuilds sequences from the raw sealed-test channel frame, and scores the sequence-eligible sealed-test cohort on the manifest's selected `sequence_length`, without retraining or changing the stored prediction semantics. The returned row count is not `len(comparison_test_rows)`: only rows with a complete lookback window are scored.

In [ ]:
comparison_prediction_values, comparison_origins = score_saved_model(
    rnn_manifest,
    COMPARISON_MODEL_PATH,
    comparison_test_rows,
    comparison_raw_test_channel_frame,
)
print(
    f"Scored {len(comparison_origins):,} sequence-eligible sealed-test rows (of "
    f"{len(comparison_test_rows):,} eligible rows) with {len(COMPARISON_TARGET_COLUMNS)} "
    "horizons using the saved RNN model."
)

In [ ]:
comparison_prediction_columns = [
    f"prediction_{target_column}" for target_column in COMPARISON_TARGET_COLUMNS
]
comparison_prediction_table = (
    comparison_origins[["timestamp", *COMPARISON_TARGET_COLUMNS]]
    .reset_index(drop=True)
    .rename(columns={"timestamp": "issue_time"})
)
comparison_prediction_table = pd.concat(
    [
        comparison_prediction_table,
        pd.DataFrame(
            comparison_prediction_values,
            columns=comparison_prediction_columns,
        ),
    ],
    axis=1,
)
comparison_prediction_table["issue_time"] = pd.to_datetime(
    comparison_prediction_table["issue_time"], utc=True
)

comparison_issue_times = comparison_prediction_table["issue_time"]
comparison_horizons = list(range(1, FORECAST_HORIZON_HOURS + 1))
comparison_horizon_labels = [f"H+{horizon:02d}" for horizon in comparison_horizons]
comparison_time_series_frames = []
for horizon in comparison_horizons:
    target_column = COMPARISON_TARGET_COLUMNS[horizon - 1]
    prediction_column = comparison_prediction_columns[horizon - 1]
    valid_times = comparison_issue_times + pd.to_timedelta(horizon, unit="h")
    comparison_time_series_frames.append(
        go.Frame(
            name=comparison_horizon_labels[horizon - 1],
            data=[
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[target_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Actual",
                    hovertemplate=(
                        "Valid time=%{x}<br>Issue time=%{customdata}<br>"
                        "Actual=%{y:.3f}<extra></extra>"
                    ),
                ),
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[prediction_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Prediction",
                    hovertemplate=(
                        "Valid time=%{x}<br>Issue time=%{customdata}<br>"
                        "Prediction=%{y:.3f}<extra></extra>"
                    ),
                ),
            ],
        )
    )
comparison_time_series_steps = [
    {
        "label": comparison_horizon_labels[horizon - 1],
        "method": "animate",
        "args": [[comparison_horizon_labels[horizon - 1]], {"mode": "immediate"}],
    }
    for horizon in comparison_horizons
]
comparison_time_series_figure = go.Figure(
    data=comparison_time_series_frames[0].data,
    frames=comparison_time_series_frames,
    layout={
        "title": "Saved RNN predictions across forecast horizons",
        "xaxis_title": "Valid time",
        "yaxis_title": "Water level",
        "hovermode": "x unified",
        "sliders": [
            {
                "active": 0,
                "currentvalue": {"prefix": "Forecast horizon: "},
                "steps": comparison_time_series_steps,
            }
        ],
    },
)
rnn_prediction_time_series_figure = comparison_time_series_figure
display(comparison_time_series_figure)

## Inspect best and worst RNN forecast windows

The following plots use the saved-model predictions and select sealed-test issue times by the RMSE calculated across every configured forecast horizon. A context window is eligible only when the target-station water-level series contains every hourly observation across the notebook's configured context window, with no imputed observations.

In [ ]:
rnn_forecast_window_figures = forecast_window_figures(
    comparison_prediction_table,
    comparison_dataset.target_context_series,
    water_level_column=f"{TARGET_STATION_ID}__water_level",
    imputed_column=f"{TARGET_STATION_ID}__imputed",
    prediction_columns=comparison_prediction_columns,
    target_columns=COMPARISON_TARGET_COLUMNS,
    horizons=comparison_horizons,
    label_prefix="RNN",
)

### Best

In [ ]:
best_rnn_forecast_window_figure = rnn_forecast_window_figures["best"]
display(best_rnn_forecast_window_figure)

### Worst

In [ ]:
worst_rnn_forecast_window_figure = rnn_forecast_window_figures["worst"]
display(worst_rnn_forecast_window_figure)

## Compare absolute and signed errors

Each box contains all sequence-eligible sealed-test errors for one horizon. Absolute-error markers reuse the current execution's per-horizon MAE/RMSE values; signed errors follow the convention `prediction - actual`.

In [ ]:
comparison_actual_values = comparison_origins[COMPARISON_TARGET_COLUMNS].to_numpy(
    dtype=float
)
signed_errors = comparison_prediction_values - comparison_actual_values
absolute_errors = np.abs(signed_errors)

absolute_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(absolute_errors),
            y=absolute_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_mae"],
            mode="markers",
            name="MAE",
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_rmse"],
            mode="markers",
            name="RMSE",
        ),
    ]
)
absolute_error_boxplot_figure.update_layout(
    title="Saved RNN absolute errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Absolute error",
)

display(absolute_error_boxplot_figure)

In [ ]:
signed_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(signed_errors),
            y=signed_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=signed_errors.mean(axis=0),
            mode="markers",
            name="Mean error",
        ),
    ]
)
signed_error_boxplot_figure.update_layout(
    title="Saved RNN signed errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Signed error (prediction - actual)",
)
signed_error_boxplot_figure.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
)
display(signed_error_boxplot_figure)

## Selected RNN architecture

Ridge's final section extracts a closed-form linear coefficient formula, and MLP's reports whether training hit its iteration cap; neither has a direct counterpart here. Instead, this section reloads the saved model and reports its fitted architecture, parameter count, and training loss curve — since there is no early-stopping/convergence criterion, the fixed epoch budget always ran to completion by design, so the loss-curve shape is the only convergence diagnostic.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown
from joblib import load as load_joblib

architecture_model = load_joblib(COMPARISON_MODEL_PATH)
architecture_module = architecture_model.module_
architecture_parameter_count = sum(
    parameter.numel() for parameter in architecture_module.parameters()
)

architecture_summary_table = pd.DataFrame(
    [
        {
            "cell_type": rnn_manifest.selected_cell_type,
            "sequence_length": rnn_manifest.selected_sequence_length,
            "hidden_size": rnn_manifest.selected_hidden_size,
            "num_layers": rnn_manifest.selected_num_layers,
            "dropout": rnn_manifest.fixed_dropout,
            "parameter_count": architecture_parameter_count,
            "epochs_trained": len(architecture_model.epoch_losses_),
        }
    ]
)
display(
    Markdown(
        f"""The saved model uses cell_type **{rnn_manifest.selected_cell_type}**,
sequence_length **{rnn_manifest.selected_sequence_length}**, hidden_size
**{rnn_manifest.selected_hidden_size}**, and num_layers
**{rnn_manifest.selected_num_layers}**."""
    )
)
display(architecture_summary_table)
print(
    "Training ran for the fixed epoch budget by design (no early stopping); "
    "the loss curve below is the only convergence diagnostic."
)

architecture_loss_curve_figure, architecture_loss_curve_axis = plt.subplots(
    figsize=(8, 4)
)
architecture_loss_curve_axis.plot(
    range(1, len(architecture_model.epoch_losses_) + 1),
    architecture_model.epoch_losses_,
)
architecture_loss_curve_axis.set_xlabel("Epoch")
architecture_loss_curve_axis.set_ylabel("Training MSE (scaled targets)")
architecture_loss_curve_axis.set_title("Saved RNN final retrain loss curve")
architecture_loss_curve_axis.grid(alpha=0.3)
plt.show()